
# Step 4 - subset spawning location dataframes etc:
Input: data from step 3 

Filter for when there are 21 days spent in a location as “used”
export
Output:  'IDLloc_spawning_21dfilter.csv'

summary.to_csv(output_path + 'nursery_use_summary.csv', index=False)


In [1]:
import xarray as xr
import pandas as pd
from datetime import datetime, timedelta
import os

from shapely.geometry import Point, Polygon as ShapelyPolygon
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
# File paths to data
repo_path = '/Users/zephyrsylvester/repos/circumpolar-connectivity-analysis/data/'
output_path = '/Users/zephyrsylvester/repos/circumpolar-connectivity-analysis/processed_data/'
fig_path = '/Users/zephyrsylvester/repos/circumpolar-connectivity-analysis/figures/'

# List of simulations and locations
sim_list = ['007', '008', '009', '010', '011', '012', '013', '014', '015', '016', '017', '018']
locations = ['BS', 'GERL', 'GP', 'MB2']


In [3]:
# file = output_path + 'processed_trajectories_sub3filt.csv'
file = output_path + 'processed_trajectories_21dfilter.csv'
filtered_df=pd.read_csv(file) 
print('raw number of larvae:', filtered_df.larval_id.nunique())
filtered_df

raw number of larvae: 11487


,N,T,date,lat,lon,release,stage,larval_id,start_year,hypothesis,IDL_loc,BS,GERL,GP,MB2,outside
0,2,0,2016-11-01,-63.602314,298.26752,0,0,00_16_1_0003,2016,h_null,BS,True,False,False,False,False
1,2,1,2016-11-02,-63.569042,298.23145,0,0,00_16_1_0003,2016,h_null,BS,True,False,False,False,False
2,2,2,2016-11-03,-63.522503,298.25543,0,0,00_16_1_0003,2016,h_null,BS,True,False,False,False,False
3,2,3,2016-11-04,-63.483067,298.28790,0,0,00_16_1_0003,2016,h_null,BS,True,False,False,False,False
4,2,4,2016-11-05,-63.438553,298.33795,0,0,00_16_1_0003,2016,h_null,BS,True,False,False,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2067655,198,175,2019-09-12,-61.923122,299.13315,10,4,00_18_1_0199,2018,h_null,BS,False,False,False,False,True
2067656,198,176,2019-09-13,-61.860650,299.25064,10,4,00_18_1_0199,2018,h_null,BS,False,False,False,False,True
2067657,198,177,2019-09-14,-61.813430,299.41430,10,4,00_18_1_0199,2018,h_null,BS,False,False,False,False,True
2067658,198,178,2019-09-15,-61.756500,299.58655,10,4,00_18_1_0199,2018,h_null,BS,False,False,False,False,True


In [4]:
# Create a summary of the number of unique larval_id's for each hypothesis, start year, and IDL_loc
summary_df = filtered_df.groupby(['hypothesis', 'start_year', 'IDL_loc'])['larval_id'].nunique().reset_index()

# Rename the column for clarity
summary_df.rename(columns={'larval_id': 'unique_larval_count'}, inplace=True)

summary_df.head()

,hypothesis,start_year,IDL_loc,unique_larval_count
0,h_dvm,2016,BS,290
1,h_dvm,2016,GERL,45
2,h_dvm,2016,GP,66
3,h_dvm,2016,MB2,60
4,h_dvm,2017,BS,184


# Extract Starting Locations

In [5]:
# Extract Summary Information
critical_columns = ['larval_id', 'date', 'lon', 'lat', 'hypothesis', 'start_year', 'IDL_loc', 'stage']
presence_columns = ['BS', 'GERL', 'GP', 'MB2','outside']

# Function to create the starting location DataFrame
def create_starting_location_df(df):
    t0_data = df[df['T'] == 0].copy()
    starting_locations = t0_data[critical_columns+presence_columns].copy()  # Use copy() to avoid SettingWithCopyWarning
    starting_locations.rename(columns={'date': 'release_date', 'lat': 'release_lat', 'lon': 'release_lon'}, inplace=True)
    return starting_locations

In [6]:
# Create the starting location DataFrame
starting_location_df = create_starting_location_df(filtered_df)

# Check It
print('number of larvae:', starting_location_df.larval_id.nunique())
# presence_summary_starting_location = summarize_presence(starting_location_df)
starting_location_df.head()

number of larvae: 11487


,larval_id,release_date,release_lon,release_lat,hypothesis,start_year,IDL_loc,stage,BS,GERL,GP,MB2,outside
0,00_16_1_0003,2016-11-01,298.26752,-63.602314,h_null,2016,BS,0,True,False,False,False,False
180,00_16_1_0004,2016-11-01,298.70044,-63.499004,h_null,2016,BS,0,True,False,False,False,False
360,00_16_1_0005,2016-11-01,298.89746,-63.201790,h_null,2016,BS,0,True,False,False,False,False
540,00_16_1_0006,2016-11-01,299.32642,-63.093323,h_null,2016,BS,0,True,False,False,False,False
720,00_16_1_0007,2016-11-01,299.74160,-62.983440,h_null,2016,BS,0,True,False,False,False,False


In [7]:
# Create a summary of the number of unique larval_id's for each hypothesis, start year, and IDL_loc
summary_sg = starting_location_df.groupby(['hypothesis', 'start_year', 'IDL_loc'])['larval_id'].nunique().reset_index()

# Rename the column for clarity
summary_sg.rename(columns={'larval_id': 'unique_larval_count'}, inplace=True)

summary_sg.head()

,hypothesis,start_year,IDL_loc,unique_larval_count
0,h_dvm,2016,BS,290
1,h_dvm,2016,GERL,45
2,h_dvm,2016,GP,66
3,h_dvm,2016,MB2,60
4,h_dvm,2017,BS,184


In [8]:
def compare_dataframes(df1, df2):
    if df1.equals(df2):
        print("The DataFrames are exactly the same.")
    else:
        print("The DataFrames are not the same.")

# Example usage:
compare_dataframes(summary_sg, summary_df)

The DataFrames are exactly the same.


In [9]:
# Save the starting locations DataFrame to a CSV file
starting_location_df.to_csv(output_path + 'IDLloc_spawning_21dfilter.csv', index=False)

# Save the location summary DataFrame to a CSV file
summary_sg.to_csv(output_path + 'nursery_use_summary.csv', index=False)

In [10]:
# # Save the starting locations DataFrame to a CSV file
# starting_location_df.to_csv(output_path + 'IDLloc_spawning_sub3filt.csv', index=False)

# # Save the location summary DataFrame to a CSV file
# summary_sg.to_csv(output_path + 'nursery_use_summary_sub3filt.csv', index=False)